In [1]:
import nltk
from tqdm import tqdm
from nltk.corpus import brown

nltk.download('brown') # Downloads the dataset

brown_words = brown.words() # Access raw text (words)

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


In [2]:
from collections import Counter

unique_words = list(set(brown_words)) # get unique words ( no repetation)

count = dict(Counter( word.lower() for word in brown_words)) # get the number of repeating of each word

normalized_count={k:min(v,1000) for k,v in count.items() if v>5 and k.isalpha()} 
#neglect words that contain special characters or numbers or words that appear less than 5 times
# set the maximum number of repeatings to 1000

exist={k: (v>10 and k.isalpha()) for k,v in count.items()}
# dictionary of words and a boolean value that is true only if it exists in normalized_count

word_ind={key:ind for ind, key in enumerate(normalized_count.keys())}
# dictionary of words and their index in the normalized_count

normalized_unique_data=[]   #list of all words in normalized_count with the constrainted repeatings
for k,v in normalized_count.items():
  normalized_unique_data+=[word_ind[k]]*v;

In [3]:
k=2   # window for the context
training_dataset=[]
for sent in brown.sents():  # for each sentense 
  count=0; # count of the sequence of previous words existed in normalized_count
  sent_exist=[exist[word.lower()] for word in sent] #check if every word existed in normalized_count
  sent_ind=[word_ind.get(word.lower(), -1) for word in sent] # index of words if existed in normalized_count, or -1
  for ind in range(len(sent_exist)-k): #checks target words
    if(sent_exist[ind]==False): #if not found
      count=0; # reset the counter
    else:
      count+=1; # otherwise increment it
    if count>k: # if a sequence of k-words is found
      if(sent_exist[ind+1] and sent_exist[ind+2]): #check if the k words after is also exist
        training_dataset.append(sent_ind[ind-k:ind+k+1]) # if so, insert the 2k+1 words in the training_dataset

In [4]:
k=2  # window for the context
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

training_dataset_torch=torch.tensor(training_dataset,dtype=int) # convert it to tensor
n1=(training_dataset_torch.shape[0]*7)//10  # 0 ... n1 training
n2=(training_dataset_torch.shape[0]*9)//10  # n1... n2 testing
dev="cuda:0"   # cuda device, set to "cpu" if no GPU found

criterion = nn.CrossEntropyLoss().to(dev) # cross entropy as a loss function

embedding_dim=100;   # each embedding is a vector of 100 elements 
neg_samples_length=10; #10 negative samples

#1 0 ... 0
output= [0]*(neg_samples_length+1)
output[0]=1;
output = torch.tensor(output).float().to(dev)

normalized_unique_data_torch=torch.tensor(normalized_unique_data,dtype=int); # to tensor

class Model(nn.Module): # module
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.embedding_context = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim) #embedding for context
        self.embedding_target = nn.Embedding(num_embeddings=num_embeddings, embedding_dim=embedding_dim)  #embedding for target

model=Model(len(normalized_count.keys()),embedding_dim).to(dev) # instance from the model
optimizer = optim.Adam(model.parameters(), lr=1e-3) #Adam optimizer


In [5]:
#training one epoch
def train(model,optimizer,criterion,training_dataset_torch,normalized_unique_data_torch,n1):
    c=0; # count of processed samples
    avg_loss=0; #accumlated loss 
    bar=tqdm(torch.randperm(n1)) # shuffle the training set and associate it to a progress bar
    for sample_ind in bar: # for each sample index
      c+=1; # increment the counter
      model.zero_grad() #reset the gradients
      sample=training_dataset_torch[sample_ind,:].to(dev) #get the sample 
      random_indices = torch.randint(0, len(normalized_unique_data), (neg_samples_length+1,)) # get the negative samples index
      samples=normalized_unique_data_torch[random_indices].to(dev) # get the negative samples
      samples[0]=sample[k] # set the first to be the target 
      context_embedding=(model.embedding_context(sample[0:k])+model.embedding_context(sample[k+1:])).mean(axis=0); 
      #calculate the average embedding of the context
        
      dot_product=torch.matmul(model.embedding_target(samples),context_embedding) 
      # dot product between the context and the negative samples and the target
    
      loss = criterion(dot_product, output) #calculate the loss function
    
      loss.backward() # backward propagation
      optimizer.step() # updated the parameters
      avg_loss+=loss.detach().sum() # uodate the accumlated loss
    
      bar.set_description("Training, Average Loss: "+str(avg_loss.item()/c)) # update the progress bar to print the average loss
    return model,avg_loss/c # return the updated loss, average loss
#testing one epoch
def test(criterion,training_dataset_torch,normalized_unique_data_torch,n1,n2):
    bar=tqdm(range(n1,n2)) # create a progress bar with the sample indices 
    c=0; # count of processed samples
    avg_loss=0; #accumlated loss 
    with torch.no_grad(): #no gradients is needed
      for sample_ind in bar: # for each sample index
        c+=1; # increment the counter
        sample=training_dataset_torch[sample_ind,:].to(dev) #get the sample 
        random_indices = torch.randint(0, len(normalized_unique_data), (neg_samples_length+1,))  # get the negative samples index
        samples=normalized_unique_data_torch[random_indices].to(dev) # get the negative samples
        samples[0]=sample[k] # set the first to be the target 
        context_embedding=(model.embedding_context(sample[0:k])+model.embedding_context(sample[k+1:])).mean(axis=0);
        #calculate the average embedding of the context
        dot_product=torch.matmul(model.embedding_target(samples),context_embedding)
        # dot product between the context and the negative samples and the target
    
        loss = criterion(dot_product, output) #calculate the loss function
    
        avg_loss+=loss.detach().sum() # uodate the accumlated loss 
    
        bar.set_description("Testing, Average Loss: "+str(avg_loss.item()/c)) # update the progress bar to print the average loss
    return avg_loss/c  # return the average loss

In [6]:
divider=1;
for epoch in range(6):
    print("epoch "+str(epoch));
    model,avg_loss=train(model,optimizer,criterion,training_dataset_torch,normalized_unique_data_torch,n1)
    test(criterion,training_dataset_torch,normalized_unique_data_torch,n1,n2)
    if epoch%2 ==1:
        divider*=3
        optimizer = optim.Adam(model.parameters(), lr=1e-3/divider)

epoch 0


Testing, Average Loss: 4.48986024064852: 100%|███████████████████████████████████| 70684/70684 [04:51<00:00, 242.88it/s]


epoch 1


Testing, Average Loss: 3.55947774602456: 100%|███████████████████████████████████| 70684/70684 [04:40<00:00, 252.07it/s]


epoch 2


...

Testing, Average Loss: 3.1977867604408354: 100%|█████████████████████████████████| 70684/70684 [06:04<00:00, 193.94it/s]


epoch 4


Testing, Average Loss: 3.1373067543574216: 100%|█████████████████████████████████| 70684/70684 [05:28<00:00, 215.20it/s]


epoch 5


Testing, Average Loss: 3.0994934320355383: 100%|█████████████████████████████████| 70684/70684 [05:21<00:00, 219.96it/s]


In [15]:
word='the'

import torch.nn.functional as F

similarities = F.cosine_similarity(model.embedding_target.weight, model.embedding_target(torch.tensor([word_ind[word]]).to(dev)), dim=1)
values, indices = torch.topk(similarities, k=10)


In [16]:
ll=list(normalized_count.keys());
for ind in indices[1:]:
    print(str(ll[ind])+" : "+ str(similarities[ind].item()*100) +" %" )

a : 75.17915964126587 %
this : 58.74977111816406 %
his : 52.31844186782837 %
their : 49.54453408718109 %
its : 46.896156668663025 %
our : 43.716275691986084 %
an : 43.0280327796936 %
any : 33.82691740989685 %
your : 28.26741933822632 %


In [64]:
word='perhaps'

import torch.nn.functional as F

similarities = F.cosine_similarity(model.embedding_target.weight, model.embedding_target(torch.tensor([word_ind[word]]).to(dev)), dim=1)
values, indices = torch.topk(similarities, k=10)

In [65]:
ll=list(normalized_count.keys());
for ind in indices[1:]:
    print(str(ll[ind])+" : "+ str(similarities[ind].item()*100) +" %" )

however : 60.04675626754761 %
although : 58.67009162902832 %
valley : 58.45990180969238 %
street : 57.14007616043091 %
once : 56.42869472503662 %
again : 56.133151054382324 %
particularly : 55.555033683776855 %
straight : 55.172377824783325 %
say : 54.94903326034546 %
